In [ ]:
import os
import json
import re
import torch
from torch.optim import AdamW
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from transformers import AutoProcessor, LayoutLMv3ForTokenClassification
from transformers import get_linear_schedule_with_warmup
from pytorch_lightning.loggers import TensorBoardLogger, WandbLogger
import numpy as np
import albumentations as A
from pathlib import Path
from difflib import SequenceMatcher 

In [ ]:
import torch
from torch.utils.data import Dataset
from pathlib import Path
from PIL import Image
import json
import numpy as np
import albumentations as A
from difflib import SequenceMatcher

class SROIEDataset(Dataset):
    def __init__(self, folder_path, processor, label_map, train=True):
        self.processor = processor 
        self.label_map = label_map 
        self.root = Path(folder_path)
        self.train = train
        
        # Define image augmentation pipeline for training
        self.transform = A.Compose([
            A.OneOf([
                A.SafeRotate(limit=5, border_mode=0, value=(255, 255, 255), p=0.5), 
                A.Perspective(scale=(0.02, 0.05), pad_val=(255, 255, 255), p=0.5), 
            ], p=0.4),
        
            A.OneOf([
                A.MotionBlur(blur_limit=5, p=0.5), 
                A.MedianBlur(blur_limit=3, p=0.5), 
                A.ImageCompression(quality_lower=70, quality_upper=100, p=0.5), 
            ], p=0.3),
        
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.OneOf([
                A.RandomShadow(p=0.5), 
                A.CLAHE(clip_limit=2.0, p=0.5), 
            ], p=0.3),
        
            A.ShiftScaleRotate(
                shift_limit=0.03, scale_limit=0.05, rotate_limit=2, 
                border_mode=0, value=(255, 255, 255), p=0.3
            ),
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['category_ids']))
        
        self.file_ids = []
        img_dir = self.root / 'img' 

        # Filter files that have both bounding boxes and entity labels
        for img_file in img_dir.glob("*.jpg"):
            fid = img_file.stem
            if (self.root / 'box' / f"{fid}.txt").exists() and \
               (self.root / 'entities' / f"{fid}.txt").exists():
                self.file_ids.append(fid)

    def __len__(self):
        return len(self.file_ids)

    def _assign_label(self, text, entities):
        """Assigns an entity label to a specific word based on string matching."""
        text_clean = text.replace(",", "").strip().lower()
        
        for key, val in entities.items():
            val_clean = str(val).replace(",", "").strip().lower()
            
            # Exact or substring match
            if text_clean in val_clean or val_clean in text_clean:
                return key.upper()
            
            # Fuzzy match for OCR errors
            if SequenceMatcher(None, text_clean, val_clean).ratio() > 0.8:
                return key.upper()
        
        return "O" # Outside/Other
        

    def __getitem__(self, idx):
        fid = self.file_ids[idx]
        
        # Load image and ground truth entities
        img = Image.open(self.root / 'img' / f"{fid}.jpg").convert("RGB")
        with open(self.root / 'entities' / f"{fid}.txt", 'r') as f:
            entities = json.load(f)

        words, boxes, word_labels = [], [], []
        
        # Parse OCR results and bounding boxes
        with open(self.root / 'box' / f"{fid}.txt", 'r', errors='ignore') as f:
            for line in f.read().splitlines():
                if not line: continue
                
                parts = line.split(",")
                if len(parts) < 9: continue                        
                
                # Extract coordinates (x1, y1, x2, y2)
                coords = [float(parts[0]), float(parts[1]), float(parts[4]), float(parts[5])]
                
                # Join remaining parts as the text content
                text = ",".join(parts[8:])
                label = self._assign_label(text, entities)
                
                # Convert to BIO format (Beginning-Entity)
                final_label = f"B-{label}" if label != "O" else "O"
                
                words.append(text)
                boxes.append(coords)
                word_labels.append(self.label_map.get(final_label, 0))

        # Apply augmentations during training
        if self.train:
            img_np = np.array(img)
            word_indices = list(range(len(words)))
            
            try:
                augmented = self.transform(
                    image=img_np, 
                    bboxes=boxes, 
                    category_ids=word_indices
                )
                
                if len(augmented['bboxes']) > 0:
                    img = Image.fromarray(augmented['image'])
                    
                    new_words, new_word_labels = [], []
                    for i in augmented['category_ids']:
                        new_words.append(words[i])
                        new_word_labels.append(word_labels[i])
                    
                    words, word_labels = new_words, new_word_labels
                    boxes = augmented['bboxes']
            except Exception:
                pass # Fallback to original if augmentation fails

        # Normalize bounding boxes to 0-1000 scale for LayoutLM
        w, h = img.size
        normalized_boxes = []
        for box in boxes:
            normalized_boxes.append([
                max(0, min(1000, int(1000 * (box[0] / w)))),                    
                max(0, min(1000, int(1000 * (box[1] / h)))),                    
                max(0, min(1000, int(1000 * (box[2] / w)))),
                max(0, min(1000, int(1000 * (box[3] / h))))
            ])

        # Prepare final encoding using the LayoutLM processor
        encoding = self.processor(
            img, 
            words, 
            boxes=normalized_boxes, 
            word_labels=word_labels,
            truncation=True,        
            padding="max_length",   
            max_length=512,
            return_tensors="pt"
        )
        
        # Remove batch dimension added by return_tensors="pt"
        encoding = {k: v.squeeze() for k, v in encoding.items()}
        
        if "labels" in encoding:
            encoding["labels"] = encoding["labels"].to(torch.long)
            
        return encoding

In [ ]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader

class SROIEDataModule(pl.LightningDataModule):
    def __init__(self, data_dir, processor, label_map, batch_size=4):
        """
        Initializes the DataModule for the SROIE dataset.
        
        Args:
            data_dir: Path to the root directory containing train/val/test splits.
            processor: The LayoutLM processor (image + text).
            label_map: Dictionary mapping string labels to integer IDs.
            batch_size: Number of samples per batch.
        """
        super().__init__()
        self.data_dir = data_dir   
        self.processor = processor 
        self.label_map = label_map 
        self.batch_size = batch_size 
    
    def setup(self, stage=None):
        """
        Splits and prepares datasets based on the current stage (fit, test, etc.).
        """
        # Set up training and validation datasets
        if stage == "fit" or stage is None:
            self.train_ds = SROIEDataset(
                f"{self.data_dir}/train", self.processor, self.label_map, train=True
            )
            self.val_ds = SROIEDataset(
                f"{self.data_dir}/val", self.processor, self.label_map, train=False
            )
        
        # Set up test dataset
        if stage == "test" or stage is None:
            self.test_ds = SROIEDataset(
                f"{self.data_dir}/test", self.processor, self.label_map, train=False
            )

    def train_dataloader(self):
        """Returns the training data loader with shuffling enabled."""
        return DataLoader(
            self.train_ds, 
            batch_size=self.batch_size, 
            shuffle=True,         
            num_workers=2,         
            pin_memory=True       
        )

    def val_dataloader(self):
        """Returns the validation data loader."""
        return DataLoader(
            self.val_ds, 
            batch_size=self.batch_size, 
            shuffle=False,         
            num_workers=2,
            pin_memory=True
        )

    def test_dataloader(self):
        """Returns the test data loader."""
        return DataLoader(
            self.test_ds, 
            batch_size=self.batch_size, 
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )

In [ ]:
import torch
import pytorch_lightning as pl
from transformers import LayoutLMv3ForTokenClassification, get_linear_schedule_with_warmup
from seqeval.metrics import f1_score, precision_score, recall_score

class LayoutLMv3Module(pl.LightningModule):
    def __init__(self, label2id, lr=2e-5):
        """
        Initializes the LightningModule for LayoutLMv3 Token Classification.
        
        Args:
            label2id: Dictionary mapping labels to IDs.
            lr: Base learning rate.
        """
        super().__init__()
        self.save_hyperparameters()
        self.label2id = label2id
        self.id2label = {v: k for k, v in label2id.items()}
        
        # Load the pre-trained LayoutLMv3 model
        self.model = LayoutLMv3ForTokenClassification.from_pretrained(
            "microsoft/layoutlmv3-base", 
            num_labels=len(label2id) 
        )
        # Buffer to store validation outputs for epoch-end metric calculation
        self.validation_step_outputs = []

    def forward(self, batch):
        """Forward pass through the model."""
        return self.model(**batch)
    
    def training_step(self, batch, batch_idx):
        """Executes a single training step and logs loss."""
        outputs = self(batch)
        loss = outputs.loss 
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        """Executes a validation step, processes predictions, and handles masks."""
        outputs = self(batch)
        val_loss = outputs.loss
        
        # Move to CPU and convert to numpy for metric calculation
        preds = outputs.logits.argmax(-1).detach().cpu().numpy()
        labels = batch["labels"].detach().cpu().numpy()
        
        true_predictions, true_labels = [], []
        for i in range(len(labels)):
            # Filter out special tokens (like padding/subwords) using mask -100
            mask = labels[i] != -100 
            true_predictions.append([self.id2label[p] for p in preds[i][mask]])
            true_labels.append([self.id2label[l] for l in labels[i][mask]])
            
        self.log("val_loss", val_loss, prog_bar=True)
        output = {"preds": true_predictions, "labels": true_labels}
        self.validation_step_outputs.append(output)
        return output
    
    def on_validation_epoch_end(self):
        """Calculates and logs aggregate metrics (F1, Precision, Recall) at the end of the epoch."""
        if not self.validation_step_outputs: return
        
        # Flatten predictions and labels across all validation batches
        all_preds = [p for out in self.validation_step_outputs for p in out["preds"]]
        all_labels = [l for out in self.validation_step_outputs for l in out["labels"]]
        
        # Calculate sequence labeling metrics
        val_f1 = f1_score(all_labels, all_preds)
        self.log("val_f1", val_f1, prog_bar=True) 
        self.log("val_precision", precision_score(all_labels, all_preds))
        self.log("val_recall", recall_score(all_labels, all_preds))
        
        # Clear buffer for the next epoch
        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        """Sets up the AdamW optimizer with differential learning rates and weight decay."""
        no_decay = ["bias", "LayerNorm.weight"]
        
        # Group parameters to apply different LR or Weight Decay
        optimizer_grouped_parameters = [
            {
                # Standard parameters with weight decay
                "params": [p for n, p in self.model.named_parameters() if not any(nd in n for nd in no_decay) and "classifier" not in n],
                "weight_decay": 0.01,
                "lr": self.hparams.lr
            },
            {
                # Bias and LayerNorm parameters without weight decay
                "params": [p for n, p in self.model.named_parameters() if any(nd in n for nd in no_decay) and "classifier" not in n],
                "weight_decay": 0.0,
                "lr": self.hparams.lr
            },
            {
                # Classifier head with a higher learning rate (10x)
                "params": [p for n, p in self.model.classifier.named_parameters()],
                "weight_decay": 0.01,
                "lr": self.hparams.lr * 10 
            }
        ]
        
        optimizer = torch.optim.AdamW(optimizer_grouped_parameters)

        # Setup learning rate scheduler with linear warmup
        total_steps = self.trainer.estimated_stepping_batches
        warmup_steps = int(total_steps * 0.1)
        
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step", 
                "frequency": 1
            }
        }

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger
from transformers import AutoProcessor

# --- Training Callbacks Configuration ---

# Saves the best model based on Validation F1 score
checkpoint_callback = ModelCheckpoint(
    monitor="val_f1",
    dirpath="checkpoints",
    filename="best-layoutlmv3-sroie",
    mode="max",
    save_top_k=1
)

# Stops training if Validation F1 does not improve for 4 consecutive epochs
early_stop_callback = EarlyStopping(
    monitor="val_f1", 
    patience=4, 
    mode="max"
)

# --- Label Mapping & Processor ---

label2id = {
    "O": 0, 
    "B-COMPANY": 1, "I-COMPANY": 2, 
    "B-DATE": 3, "I-DATE": 4, 
    "B-ADDRESS": 5, "I-ADDRESS": 6, 
    "B-TOTAL": 7, "I-TOTAL": 8
}

# OCR is set to False because we are using the bounding boxes provided in the dataset
processor = AutoProcessor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)
DATA_ROOT = "/kaggle/input/datasets/maxbegal/dataset-layoutlm/data"

# --- Data & Model Initialization ---

dm = SROIEDataModule(DATA_ROOT, processor, label2id, batch_size=4)
model = LayoutLMv3Module(label2id=label2id)

# --- Trainer Setup ---

trainer = pl.Trainer(
    max_epochs=20, 
    accelerator="gpu",
    devices=1,
    precision="16-mixed", # Uses Mixed Precision for faster training and lower VRAM usage
    logger=TensorBoardLogger("logs/", name="layoutlmv3_final"),
    callbacks=[checkpoint_callback, early_stop_callback]
)

# Start training process
trainer.fit(model, dm)

# --- Post-Training / Saving ---

# Load the best weights discovered during training
if checkpoint_callback.best_model_path:
    print(f"Loading best model from: {checkpoint_callback.best_model_path}")
    model = LayoutLMv3Module.load_from_checkpoint(
        checkpoint_callback.best_model_path, 
        label2id=label2id
    )

# Export the final model and processor for inference
model.model.save_pretrained("layoutlmv3_sroie_final")
processor.save_pretrained("layoutlmv3_sroie_final")

print("✅ Training complete. Model saved to 'layoutlmv3_sroie_final'")

preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

The image processor of type `LayoutLMv3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


vocab.json: 0.00B [00:00, ?B/s]

...

Загрузка лучшей модели из: /kaggle/working/checkpoints/best-layoutlmv3-sroie.ckpt


Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                                | Status     | 
-----------------------------------+------------+-
layoutlmv3.embeddings.position_ids | UNEXPECTED | 
classifier.bias                    | MISSING    | 
classifier.weight                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Обучение завершено, модель сохранена в 'layoutlmv3_sroie_final'


БЛОГ ИНФЕРЕНСА

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

def evaluate_model(pl_model, dataloader, label2id):
    """
    Evaluates the LayoutLMv3 model performance on a given dataloader.
    
    Args:
        pl_model: The trained LayoutLMv3Module (LightningModule).
        dataloader: PyTorch DataLoader (usually test_dataloader).
        label2id: Mapping of labels to integer IDs.
    """
    id2label = {v: k for k, v in label2id.items()}
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Extract the underlying Transformers model and move to device
    model = pl_model.model.to(device)
    model.eval()

    true_labels = []
    pred_labels = []

    print(f"🧐 Starting evaluation on {len(dataloader.dataset)} samples...")
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            # Move inputs to device, exclude labels for the forward pass
            inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            labels = batch["labels"].cpu().numpy() 

            outputs = model(**inputs)
            
            # Get predictions by taking the argmax of the logits
            predictions = outputs.logits.argmax(-1).detach().cpu().numpy()

            for i in range(len(labels)):
                # Mask out special tokens and padding (labeled as -100)
                mask = labels[i] != -100 
                
                # Convert numeric IDs back to string labels for seqeval
                true_seq = [id2label.get(l, "O") for l in labels[i][mask]]
                pred_seq = [id2label.get(p, "O") for p in predictions[i][mask]]
                
                if len(true_seq) > 0:
                    true_labels.append(true_seq)
                    pred_labels.append(pred_seq)

    print("\n" + "="*50)
    print("📊 TESTING RESULTS (SROIE):")
    print("="*50)
    
    if not true_labels:
        print("❌ Error: No labels were collected. Check the -100 mask logic.")
        return

    # Calculate overall metrics
    f1 = f1_score(true_labels, pred_labels)
    precision = precision_score(true_labels, pred_labels)
    recall = recall_score(true_labels, pred_labels)

    print(f"✅ F1-Score:  {f1:.4f}")
    print(f"✅ Precision: {precision:.4f}")
    print(f"✅ Recall:    {recall:.4f}")
    print("\n" + "-"*50)
    
    # Detailed report per entity category
    print(classification_report(true_labels, pred_labels))

# Prepare the data module splits
dm.setup() 

# Get the test dataloader
test_loader = dm.test_dataloader()

# Run the evaluation
evaluate_model(model, test_loader, label2id)

/tmp/ipykernel_55/3849634274.py:38: UserWarning: Argument(s) 'value' are not valid for transform SafeRotate
  A.SafeRotate(limit=5, border_mode=0, value=(255, 255, 255), p=0.5), # Безопасный поворот
/tmp/ipykernel_55/3849634274.py:39: UserWarning: Argument(s) 'pad_val' are not valid for transform Perspective
  A.Perspective(scale=(0.02, 0.05), pad_val=(255, 255, 255), p=0.5), # Искажение перспективы
/tmp/ipykernel_55/3849634274.py:46: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=70, quality_upper=100, p=0.5), # Артефакты сжатия
/tmp/ipykernel_55/3849634274.py:57: UserWarning: Argument(s) 'value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRotate(


🧐 Запускаем оценку на 168 примерах...


100%|██████████| 42/42 [00:24<00:00,  1.73it/s]



📊 РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ (SROIE):
✅ F1-Score:  0.9409
✅ Precision: 0.9395
✅ Recall:    0.9423

--------------------------------------------------
              precision    recall  f1-score   support

     ADDRESS       0.91      0.96      0.93       517
     COMPANY       0.95      0.94      0.94       241
        DATE       0.98      0.97      0.98       795
       TOTAL       0.90      0.88      0.89       457

   micro avg       0.94      0.94      0.94      2010
   macro avg       0.93      0.94      0.94      2010
weighted avg       0.94      0.94      0.94      2010

